# Client SC — Contract Management

Interactive notebook for all `Client` smart contract public functions.

| Function | Caller | Description |
|---|---|---|
| `transfer` | client | Transfer DataCap to provider, optionally completing the deal |
| `isDataSizeMatching` | deal Validator | Check active allocation size vs expected deal size |
| `claimsTerminatedEarly` | TERMINATION_ORACLE | Mark Filecoin claims as terminated |
| `getClientAllocationIdsPerDeal` | anyone | Query allocation/claim IDs for a deal |
| `terminatedClaims` | anyone | Check if a claim is marked terminated |
| `grantRole` / `revokeRole` | admin | Manage access control |

Contract addresses and keys are read from `filecoin-boost/scripts/porep-market/.env`.

> **Note:** `transfer()` requires CBOR-encoded `DataCapTypes.TransferParams` (Filecoin built-in actor call).
> On devnet this is exercised via the happy path notebook via `Client.transfer()`.

## 0. Setup

In [78]:
import builtins as _builtins
import json
import logging
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from web3 import Web3
from web3.exceptions import ContractCustomError
from web3.middleware import ExtraDataToPOAMiddleware

# ── .env ──────────────────────────────────────────────────────────────────────
BOOST_ENV = Path.home() / "Forked/filecoin-boost/scripts/porep-market/.env"
if BOOST_ENV.exists():
    load_dotenv(BOOST_ENV, override=True)
    print(f"Loaded .env from {BOOST_ENV}")
else:
    load_dotenv(override=True)
    print(f"WARNING: {BOOST_ENV} not found")

# ── CONFIG ────────────────────────────────────────────────────────────────────
_key = os.getenv("PRIVATE_KEY_TEST", os.getenv("PRIVATE_KEY", ""))
CONFIG = {
    "rpc_url":       os.getenv("RPC_URL", "http://127.0.0.1:1234/rpc/v1"),
    "private_key":   _key,
    "client_sc":     os.getenv("CLIENT_CONTRACT", ""),
    "porep_market":  os.getenv("POREP_MARKET", ""),
}

_addr_keys = ("client_sc", "porep_market")
for _k in _addr_keys:
    if CONFIG[_k]:
        CONFIG[_k] = Web3.to_checksum_address(CONFIG[_k])

if not CONFIG["client_sc"]:
    print("WARNING: CLIENT_CONTRACT not set — run 02_deploy.sh first")
else:
    print("Contract address loaded.")

print("\n── CONFIG ──────────────────────────────────────")
for _k, _v in CONFIG.items():
    print(f"  {_k:<25} {_v}")

# ── ABI dir ───────────────────────────────────────────────────────────────────
_nb = globals().get("__vsc_ipynb_file__", "")
ABI_DIR = Path(_nb).parent.parent / "abis" if _nb else BOOST_ENV.parent / "porep-market" / "abis"
assert ABI_DIR.exists(), f"ABI dir not found: {ABI_DIR}"
print(f"\nABI dir : {ABI_DIR}")

def load_abi(name: str):
    with open(ABI_DIR / f"{name}.json") as f:
        return json.load(f)

# ── Logger ────────────────────────────────────────────────────────────────────
_log_dir = Path(_nb).parent / "logs" if _nb else Path.home() / "porep-market-logs"
_log_dir.mkdir(parents=True, exist_ok=True)
if not hasattr(_builtins, "_client_sc_log"):
    _builtins._client_sc_log = _log_dir / f"client-sc-{datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
LOG_FILE = _builtins._client_sc_log

logger = logging.getLogger("porep.client")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    _fh = logging.FileHandler(LOG_FILE, mode="a")
    _fh.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-5s  %(message)s"))
    _ch = logging.StreamHandler()
    _ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(_fh)
    logger.addHandler(_ch)
log  = logger.info
logw = logger.warning
loge = logger.error
log(f"=== Session log: {LOG_FILE} ===")

=== Session log: /home/rafal/Projects/porep-market/notebooks/logs/client-sc-20260327-150905.log ===


Loaded .env from /home/rafal/Forked/filecoin-boost/scripts/porep-market/.env
Contract address loaded.

── CONFIG ──────────────────────────────────────
  rpc_url                   http://127.0.0.1:1234/rpc/v1
  private_key               0xae9de76ef8e77e3be75e7ae9de9adf6df4e35df6e39efce1ae9ee1dddddb6edd
  client_sc                 0xe08b47073aC3882bAad456AB98B2f3A1F8Ec0582
  porep_market              0x3F25Ad304C20eB440E084657ead68A77dBa68265

ABI dir : /home/rafal/Projects/porep-market/abis


In [79]:
w3 = Web3(Web3.HTTPProvider(CONFIG["rpc_url"]))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
assert w3.is_connected(), f"Cannot connect to {CONFIG['rpc_url']}"
chain_id = w3.eth.chain_id
log(f"Connected  chain_id={chain_id}  latest_block={w3.eth.block_number}")

Connected  chain_id=31415926  latest_block=3877


In [80]:
acct = w3.eth.account.from_key(CONFIG["private_key"]) if CONFIG["private_key"] else None
log(f"Account : {acct.address if acct else 'NOT SET'}")
bal = w3.eth.get_balance(acct.address) if acct else 0
log(f"Balance : {w3.from_wei(bal, 'ether')} FIL")

Account : 0x7C8c2A74cBf81395A156746e62d8FaE1071f1b7b
Balance : 40999.985666579811573332 FIL


In [81]:
client_sc = w3.eth.contract(address=CONFIG["client_sc"], abi=load_abi("Client"))
log("Client SC contract loaded.")

# ── Error map ─────────────────────────────────────────────────────────────────
_error_map: dict[str, str] = {}
for _abi_file in ABI_DIR.glob("*.json"):
    try:
        for _entry in json.load(open(_abi_file)):
            if _entry.get("type") == "error":
                _sig = _entry["name"] + "(" + ",".join(i["type"] for i in _entry.get("inputs", [])) + ")"
                _error_map["0x" + w3.keccak(text=_sig).hex()[:8]] = _sig
    except Exception:
        pass

def decode_custom_error(data: str) -> str:
    return _error_map.get(data[:10], f"unknown error {data[:10]}")

def send_tx(fn, account, value=0):
    nonce = w3.eth.get_transaction_count(account.address)
    log(f"  → {fn.fn_name}  from={account.address}  nonce={nonce}")
    try:
        tx = fn.build_transaction({
            "from": account.address,
            "nonce": nonce,
            "gasPrice": w3.eth.gas_price,
            "value": value,
            "chainId": chain_id,
        })
    except ContractCustomError as e:
        err = decode_custom_error(e.data)
        loge(f"  ✗ estimate_gas reverted: {err}")
        raise RuntimeError(f"estimate_gas reverted: {err}") from e
    signed = account.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    log(f"  ⏳ sent tx={tx_hash.hex()}")
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    if receipt["status"] == 1:
        log(f"  ✅ success  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
    else:
        loge(f"  ❌ reverted  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
        raise RuntimeError(f"Transaction reverted: {tx_hash.hex()}")
    return receipt

log(f"Helpers ready. {len(_error_map)} error selectors loaded.")

Client SC contract loaded.
Helpers ready. 110 error selectors loaded.


---
## 1. Contract constants & roles

In [82]:
admin_role       = client_sc.functions.DEFAULT_ADMIN_ROLE().call()
upgrader_role    = client_sc.functions.UPGRADER_ROLE().call()
term_oracle_role = client_sc.functions.TERMINATION_ORACLE().call()
upgrade_ver      = client_sc.functions.UPGRADE_INTERFACE_VERSION().call()

log(f"DEFAULT_ADMIN_ROLE   : {admin_role.hex()}")
log(f"UPGRADER_ROLE        : {upgrader_role.hex()}")
log(f"TERMINATION_ORACLE   : {term_oracle_role.hex()}")
log(f"UPGRADE_INTERFACE_VERSION : {upgrade_ver}")

address_to_check = acct.address  # change to any address
log("")
for role_name, role_bytes in [
    ("DEFAULT_ADMIN_ROLE",  admin_role),
    ("UPGRADER_ROLE",       upgrader_role),
    ("TERMINATION_ORACLE",  term_oracle_role),
]:
    has = client_sc.functions.hasRole(role_bytes, address_to_check).call()
    log(f"  {address_to_check} has {role_name}: {has}")

DEFAULT_ADMIN_ROLE   : 0000000000000000000000000000000000000000000000000000000000000000
UPGRADER_ROLE        : 189ab7a9244df0848122154315af71fe140f3db0fe014031783b0946b8c9d2e3
TERMINATION_ORACLE   : c368dd69a1791df65cd13e566a3c9de1268c66b726dff83916d52cec1b11e292
UPGRADE_INTERFACE_VERSION : 5.0.0

  0x7C8c2A74cBf81395A156746e62d8FaE1071f1b7b has DEFAULT_ADMIN_ROLE: True
  0x7C8c2A74cBf81395A156746e62d8FaE1071f1b7b has UPGRADER_ROLE: True
  0x7C8c2A74cBf81395A156746e62d8FaE1071f1b7b has TERMINATION_ORACLE: True


---
## 2. Query allocations for a deal

`getClientAllocationIdsPerDeal(dealId)` returns the Filecoin allocation/claim IDs
that have been registered for the deal via `transfer()`.

In [83]:
DEAL_ID = 1  # ← set deal ID

allocation_ids = client_sc.functions.getClientAllocationIdsPerDeal(DEAL_ID).call()
log(f"getClientAllocationIdsPerDeal({DEAL_ID}) → {len(allocation_ids)} allocation(s)")
for aid in allocation_ids:
    log(f"  allocationId : {aid}")

getClientAllocationIdsPerDeal(1) → 0 allocation(s)


---
## 3. Check terminated claims

In [85]:
CLAIM_ID = 1  # ← set claim ID (uint64)

terminated = client_sc.functions.terminatedClaims(CLAIM_ID).call()
log(f"terminatedClaims({CLAIM_ID}) : {terminated}")

terminatedClaims(1) : False


---
## 4. Check data size matching _(Validator only)_

`isDataSizeMatching(dealId)` queries the Filecoin VerifReg actor to check if the
active claim size matches the deal's expected size. It also prunes expired/terminated
allocation IDs from internal storage.

**Caller:** Must be the deal's Validator contract address.

In [87]:
# On devnet the deployer key can call this if it is set as the Validator.
# In production this is called by the Validator contract during settlement.
CHECK_DEAL_ID = 1  # ← set deal ID

receipt = send_tx(client_sc.functions.isDataSizeMatching(CHECK_DEAL_ID), acct)

# isDataSizeMatching returns a bool — decode from return data
result_data = w3.eth.call({
    "to": CONFIG["client_sc"],
    "data": client_sc.encodeABI(fn_name="isDataSizeMatching", args=[CHECK_DEAL_ID]),
    "from": acct.address,
})
is_matching = bool(int(result_data.hex(), 16))
log(f"isDataSizeMatching({CHECK_DEAL_ID}) : {is_matching}")

  → isDataSizeMatching  from=0x7C8c2A74cBf81395A156746e62d8FaE1071f1b7b  nonce=77
  ✗ estimate_gas reverted: ValidatorNotSet(uint256)


RuntimeError: estimate_gas reverted: ValidatorNotSet(uint256)

---
## 5. Transfer DataCap

`transfer(TransferParams params, uint256 dealId, bool dealCompleted)` is the core
Client SC entrypoint. It:

1. Deserializes CBOR-encoded `TransferParams` (FRC-46 operator data)
2. Verifies allocations/claims against the Filecoin VerifReg actor
3. Calls `DataCap.transfer()` built-in actor
4. If `dealCompleted=true` → calls `PoRepMarket.completeDeal()`

**Caller:** The client wallet that proposed the deal.

> **This requires CBOR-encoded Filecoin actor data** — not a plain Solidity call.
> In normal operation it is triggered by the Filecoin DataCap transfer flow.
> The cell below shows the minimal structure; adapt `operator_data` for your allocation.

In [ ]:
# pip install cbor2
import cbor2

TRANSFER_DEAL_ID   = DEAL_ID   # ← set deal ID
DEAL_COMPLETED     = True      # ← True to mark deal complete after transfer
PROVIDER_ACTOR_ID  = 1000      # ← Filecoin actor ID of the SP

# TransferParams structure (FRC-46):
#   to       : Filecoin address of the provider (f0<actorId>)
#   amount   : DataCap amount (attoFIL units)
#   operator_data : CBOR([allocations_array, claims_array])
#     allocations_array : [[provider_id, size], ...]
#     claims_array      : [[provider_id, claim_id], ...]  (empty for new deal)


# Build minimal operator_data for a new allocation (no claims)
allocation_size = 536870912  # ← must match deal size
allocations     = [[PROVIDER_ACTOR_ID, allocation_size]]  # [provider_id, size]
claims          = []  # no claim extensions for a new deal
operator_data   = cbor2.dumps([allocations, claims])

# f0<actorId> encoded as Filecoin ID address bytes: 0x00 + varint(actorId)
def encode_fil_id_addr(actor_id: int) -> bytes:
    """Encode a Filecoin f0 (ID) address as bytes."""
    result = []
    n = actor_id
    while n >= 0x80:
        result.append((n & 0x7f) | 0x80)
        n >>= 7
    result.append(n)
    return bytes([0x00]) + bytes(result)  # protocol 0 = ID address

provider_addr_bytes = encode_fil_id_addr(PROVIDER_ACTOR_ID)

# TransferParams tuple: (to_bytes, amount_bytes, operator_data_bytes)
# amount in DataCap units (1 DataCap = 1e18 attoFIL)
datacap_amount = allocation_size  # 1:1 DataCap bytes to allocation bytes
amount_bytes   = datacap_amount.to_bytes(32, "big")  # BigInt encoding

transfer_params = (
    provider_addr_bytes,   # to (bytes)
    amount_bytes,          # amount (bytes — BigInt)
    operator_data,         # operatorData (bytes — CBOR)
)

log(f"TransferParams built — provider={PROVIDER_ACTOR_ID}  size={allocation_size:,}")
log(f"  to (hex)         : {provider_addr_bytes.hex()}")
log(f"  operator_data    : {operator_data.hex()}")

receipt = send_tx(
    client_sc.functions.transfer(transfer_params, TRANSFER_DEAL_ID, DEAL_COMPLETED),
    acct,
)
ev_logs = client_sc.events.DatacapSpent().process_receipt(receipt)
if ev_logs:
    ev = ev_logs[0]["args"]
    log(f"DatacapSpent  client={ev['client']}  amount={ev['amount']}")


IndentationError: unexpected indent (2901678198.py, line 17)

---
## 6. Mark claims terminated early _(Termination Oracle only)_

`claimsTerminatedEarly(uint64[] claims)` is called by the **TERMINATION_ORACLE** role holder
to flag Filecoin claims as terminated before their natural expiry.
This affects `isDataSizeMatching()` — terminated claims are excluded from the active size sum.

In [90]:
term_oracle_role = client_sc.functions.TERMINATION_ORACLE().call()
is_oracle        = client_sc.functions.hasRole(term_oracle_role, acct.address).call()
log(f"Account has TERMINATION_ORACLE role: {is_oracle}")

TERMINATED_CLAIM_IDS = [1, 2, 3]  # ← list of claim IDs (uint64) to mark terminated

if is_oracle:
    receipt = send_tx(client_sc.functions.claimsTerminatedEarly(TERMINATED_CLAIM_IDS), acct)
    log(f"Marked {len(TERMINATED_CLAIM_IDS)} claim(s) as terminated: {TERMINATED_CLAIM_IDS}")
    # verify
    for cid in TERMINATED_CLAIM_IDS:
        status = client_sc.functions.terminatedClaims(cid).call()
        log(f"  terminatedClaims({cid}) : {status}")
else:
    logw("Skipped — account does not have TERMINATION_ORACLE role")

Account has TERMINATION_ORACLE role: True
  → claimsTerminatedEarly  from=0x7C8c2A74cBf81395A156746e62d8FaE1071f1b7b  nonce=77
  ⏳ sent tx=cad9395485648359d1bf2dff683f948a3c0b2ab6ddab73ef2ce81ef7aa78fd71
  ✅ success  tx=cad9395485648359d1bf2dff683f948a3c0b2ab6ddab73ef2ce81ef7aa78fd71  gas_used=11950478
Marked 3 claim(s) as terminated: [1, 2, 3]
  terminatedClaims(1) : True
  terminatedClaims(2) : True
  terminatedClaims(3) : True


---
## 7. Admin — grant / revoke roles

In [ ]:
# Grant or revoke a role on an account
ROLE_TO_GRANT  = client_sc.functions.TERMINATION_ORACLE().call()  # ← role bytes32
TARGET_ACCOUNT = acct.address  # ← account to grant/revoke
ACTION         = "grant"       # ← "grant" or "revoke"

before = client_sc.functions.hasRole(ROLE_TO_GRANT, TARGET_ACCOUNT).call()
log(f"hasRole before : {before}")

if ACTION == "grant":
    receipt = send_tx(client_sc.functions.grantRole(ROLE_TO_GRANT, TARGET_ACCOUNT), acct)
else:
    receipt = send_tx(client_sc.functions.revokeRole(ROLE_TO_GRANT, TARGET_ACCOUNT), acct)

after = client_sc.functions.hasRole(ROLE_TO_GRANT, TARGET_ACCOUNT).call()
log(f"hasRole after  : {after}")